In [ ]:
import os
import tensorflow as tf

# Make only GPU 7 visible to TensorFlow
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

# List the visible GPUs (GPU 7 will appear as GPU 0 to TensorFlow)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Allow memory growth instead of setting a fixed limit
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f"Using GPU 3 (seen as GPU 0 by TF): {len(gpus)} physical, {len(logical_gpus)} logical GPUs")
    except RuntimeError as e:
        print("TF GPU setup error:", e)
else:
    print("No GPU available. Running on CPU.")

In [ ]:
import os
import cv2
import numpy as np
import pywt
import scipy.stats
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from skimage.feature import graycomatrix, graycoprops
from tensorflow.keras.utils import to_categorical
import umap
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.tree import DecisionTreeClassifier, _tree, plot_tree
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import itertools
import tqdm
from transformers import TFViTModel, ViTFeatureExtractor


In [ ]:
# --- CONFIGURATION ---
IMG_SIZE = (224, 224)
WAVELET = 'db1'

TRAIN_DIR = '/home/D13K48009/Clara/Dataset/MES Mixed Data' # <-- Change this to your training dataset path
TEST_DIR  = '/home/D13K48009/Clara/Dataset/MES_Colonoscopy Public Dataset' # <-- Change this to your testing dataset path

# --- WAVELET + GLCM FEATURE EXTRACTORS ---
def extract_wavelet_stats(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    coeffs2 = pywt.dwt2(gray, WAVELET)
    LL, (LH, HL, HH) = coeffs2
    def stats(subband):
        return [
            np.mean(subband),
            np.std(subband),
            np.var(subband),
            scipy.stats.entropy(np.abs(subband.flatten()) + 1e-6)
        ]
    features = []
    for band in [LL, LH, HL, HH]:
        features.extend(stats(band))
    hh_energy = np.sum(np.square(HH))
    features.append(hh_energy)
    return features

def extract_glcm_features_extended(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    angles = [0, np.pi/4, np.pi/2]
    glcm = graycomatrix(gray, distances=[5], angles=angles, levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').mean()
    dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    return [contrast, dissimilarity, homogeneity]

# --- REUSABLE DATA LOADER (MODIFIED) ---
def load_dataset(folder_path, crop=False, crop_box=(30, 430, 200, 550)):
    """
    If crop=True, apply ROI cropping before resizing.
    crop_box = (y1, y2, x1, x2) in pixel coordinates of the ORIGINAL image.
    """
    X_img, X_feat, y_label, img_paths = [], [], [], []
    y1, y2, x1, x2 = crop_box

    for label in os.listdir(folder_path):
        label_path = os.path.join(folder_path, label)
        if not os.path.isdir(label_path):
            continue

        for fname in os.listdir(label_path):
            img_path = os.path.join(label_path, fname)
            img = cv2.imread(img_path)
            if img is None:
                continue

            # BGR -> RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # --- train-only crop ---
            if crop:
                H, W = img.shape[:2]
                # Clamp to valid bounds (basic safety)
                yy1, yy2 = max(0, y1), min(y2, H)
                xx1, xx2 = max(0, x1), min(x2, W)
                if (yy2 > yy1) and (xx2 > xx1):
                    roi = img[yy1:yy2, xx1:xx2]
                else:
                    # If the box is out of bounds, fall back to full image
                    roi = img
            else:
                roi = img

            # Resize AFTER (optional) crop
            resized = cv2.resize(roi, IMG_SIZE)

            # --- feature extraction on the resized image ---
            wavelet_feats = extract_wavelet_stats(resized)
            glcm_feats    = extract_glcm_features_extended(resized)
            combined      = wavelet_feats + glcm_feats

            X_img.append(resized)
            X_feat.append(combined)
            y_label.append(label)
            img_paths.append(img_path)

    return np.array(X_img), np.array(X_feat), np.array(y_label), np.array(img_paths)

# --- LOAD TRAINING + TEST SET (MODIFIED CALLS) ---
# Train = cropped, Test = NOT cropped
X_img_train_raw, X_feat_train_raw, y_train_label, img_paths_train = load_dataset(
    TRAIN_DIR, crop=True, crop_box=(30, 430, 200, 550)
)
X_img_test_raw,  X_feat_test_raw,  y_test_label,  img_paths_test  = load_dataset(
    TEST_DIR, crop=False
)

# --- NORMALIZE IMAGES ---
X_img_train = X_img_train_raw.astype(np.float32) / 255.0
X_img_test  = X_img_test_raw.astype(np.float32) / 255.0

# --- LABEL ENCODING ---
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_label)
y_test_encoded  = le.transform(y_test_label)
y_train_cat = to_categorical(y_train_encoded, num_classes=len(le.classes_))
y_test_cat  = to_categorical(y_test_encoded,  num_classes=len(le.classes_))

# --- FEATURE SCALING ---
scaler = StandardScaler()
X_feat_train_scaled = scaler.fit_transform(X_feat_train_raw)
X_feat_test_scaled  = scaler.transform(X_feat_test_raw)

# --- PLOT TRAIN CLASS DISTRIBUTION (Before SMOTE) ---
plt.figure(figsize=(8, 4))
plt.title("Training Set Class Distribution (Before SMOTE)")
plt.bar(*np.unique(y_train_encoded, return_counts=True), tick_label=le.classes_)
plt.xlabel("Class")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.show()

# --- PLOT TEST CLASS DISTRIBUTION ---
plt.figure(figsize=(8, 4))
plt.title("Testing Set Class Distribution")
plt.bar(*np.unique(y_test_encoded, return_counts=True), tick_label=le.classes_, color='orange')
plt.xlabel("Class")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.show()

# --- APPLY SMOTE TO TRAINING SET ---
smote = SMOTE(random_state=42)
X_feat_train_bal, y_train_bal = smote.fit_resample(X_feat_train_scaled, y_train_encoded)

# --- PLOT TRAIN CLASS DISTRIBUTION (After SMOTE) ---
plt.figure(figsize=(8, 4))
plt.title("Training Set Class Distribution (After SMOTE)")
plt.bar(*np.unique(y_train_bal, return_counts=True), tick_label=le.classes_, color='green')
plt.xlabel("Class")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.show()

# --- MAP BALANCED FEATURES TO REAL IMAGES ---
X_img_train_bal, img_paths_train_bal = [], []
for feat, label in zip(X_feat_train_bal, y_train_bal):
    dists = np.linalg.norm(X_feat_train_scaled[y_train_encoded == label] - feat, axis=1)
    idx = np.where(y_train_encoded == label)[0][np.argmin(dists)]
    X_img_train_bal.append(X_img_train[idx])
    img_paths_train_bal.append(img_paths_train[idx])
X_img_train_bal = np.array(X_img_train_bal, dtype=np.float32)
y_train_cat_bal = to_categorical(y_train_bal, num_classes=len(le.classes_))

# --- UMAP PROJECTION (fit on training, apply to test) ---
umap_reducer = umap.UMAP(n_neighbors=10, min_dist=0.05, n_components=2, metric='euclidean', random_state=42)
X_train_umap = umap_reducer.fit_transform(X_feat_train_bal)
X_test_umap  = umap_reducer.transform(X_feat_test_scaled)

# --- SHAPE SUMMARY ---
print(f"X_img_train_bal: {X_img_train_bal.shape}, X_img_test: {X_img_test.shape}")
print(f"X_feat_train_bal: {X_feat_train_bal.shape}, X_feat_test_scaled: {X_feat_test_scaled.shape}")
print(f"X_train_umap: {X_train_umap.shape}, X_test_umap: {X_test_umap.shape}")
print(f"y_train_cat_bal: {y_train_cat_bal.shape}, y_test_cat: {y_test_cat.shape}")


In [ ]:
import matplotlib.pyplot as plt

# --- Crop box (for training only) ---
y1, y2, x1, x2 = 30, 430, 200, 550

# --- Show training example (CROPPED) ---
img_train = cv2.imread(img_paths_train[0])
img_train = cv2.cvtColor(img_train, cv2.COLOR_BGR2RGB)
cropped_train = img_train[y1:y2, x1:x2]
resized_train = cv2.resize(cropped_train, IMG_SIZE)

# --- Show testing example (NOT CROPPED) ---
img_test = cv2.imread(img_paths_test[0])
img_test = cv2.cvtColor(img_test, cv2.COLOR_BGR2RGB)
# No crop for test:
cropped_test = img_test.copy()                       # keep same image for middle panel
resized_test = cv2.resize(img_test, IMG_SIZE)        # resize full image

# --- Plot side-by-side comparison ---
fig, axs = plt.subplots(2, 3, figsize=(12, 6))

# Row 1: Training example
axs[0, 0].imshow(img_train)
axs[0, 0].set_title("Train - Original")
axs[0, 1].imshow(cropped_train)
axs[0, 1].set_title("Train - Cropped")
axs[0, 2].imshow(resized_train)
axs[0, 2].set_title("Train - Resized 224x224")

# Row 2: Testing example (no crop)
axs[1, 0].imshow(img_test)
axs[1, 0].set_title("Test - Original")
axs[1, 1].imshow(cropped_test)
axs[1, 1].set_title("Test - No Crop (same as original)")
axs[1, 2].imshow(resized_test)
axs[1, 2].set_title("Test - Resized 224x224 (No Crop)")

# Formatting
for ax in axs.flatten():
    ax.axis('off')
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Rescale to 0–255 for visualization purposes ---
train_original = X_img_train * 255.0
test_original  = X_img_test * 255.0

# --- Flatten pixel values across all channels and images ---
train_pixels_orig = train_original.flatten()
test_pixels_orig  = test_original.flatten()

train_pixels_norm = X_img_train.flatten()
test_pixels_norm  = X_img_test.flatten()

# --- Plot original pixel distributions (0–255) ---
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(train_pixels_orig, bins=50, color='gray', alpha=0.7, label='Train')
plt.hist(test_pixels_orig,  bins=50, color='red', alpha=0.5, label='Test')
plt.title("Pixel Value Distribution [0–255]")
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.legend()

# --- Plot normalized pixel distributions (0–1) ---
plt.subplot(1, 2, 2)
plt.hist(train_pixels_norm, bins=50, color='blue', alpha=0.7, label='Train')
plt.hist(test_pixels_norm,  bins=50, color='orange', alpha=0.5, label='Test')
plt.title("Pixel Value Distribution After Normalization [0–1]")
plt.xlabel("Pixel Value")
plt.ylabel("Frequency")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Use handcrafted feature names
feature_names = [
    'LL_mean', 'LL_std', 'LL_var', 'LL_entropy',
    'LH_mean', 'LH_std', 'LH_var', 'LH_entropy',
    'HL_mean', 'HL_std', 'HL_var', 'HL_entropy',
    'HH_mean', 'HH_std', 'HH_var', 'HH_entropy',
    'GLCM_contrast', 'GLCM_dissimilarity', 'GLCM_homogeneity',
    'HH_energy'
]

# Create raw DataFrames
df_train_raw = pd.DataFrame(X_feat_train_raw[:, :20], columns=feature_names)
df_test_raw  = pd.DataFrame(X_feat_test_raw[:, :20],  columns=feature_names)

# Standardize using same scaler
scaler = StandardScaler()
df_train_scaled = pd.DataFrame(scaler.fit_transform(df_train_raw), columns=feature_names)
df_test_scaled  = pd.DataFrame(scaler.transform(df_test_raw),  columns=feature_names)

# Plot 1: Train - Before Scaling
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_train_raw)
plt.title("Train Set - Handcrafted Features (Before Standardization)")
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 2: Train - After Scaling
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_train_scaled)
plt.title("Train Set - Handcrafted Features (After Standardization)")
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 3: Test - Before Scaling
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_test_raw)
plt.title("Test Set - Handcrafted Features (Before Standardization)")
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 4: Test - After Scaling
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_test_scaled)
plt.title("Test Set - Handcrafted Features (After Standardization)")
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Handcrafted feature names
feature_names = [
    'LL_mean', 'LL_std', 'LL_var', 'LL_entropy',
    'LH_mean', 'LH_std', 'LH_var', 'LH_entropy',
    'HL_mean', 'HL_std', 'HL_var', 'HL_entropy',
    'HH_mean', 'HH_std', 'HH_var', 'HH_entropy',
    'HH_energy', 'GLCM_contrast', 'GLCM_dissimilarity', 'GLCM_homogeneity'
]

# Prepare raw DataFrames
X_feats_train = np.array(X_feat_train_raw, dtype=np.float32)
X_feats_test  = np.array(X_feat_test_raw, dtype=np.float32)
df_train_raw = pd.DataFrame(X_feats_train, columns=feature_names)
df_test_raw  = pd.DataFrame(X_feats_test,  columns=feature_names)

# Standardize using same scaler (like in real pipeline)
scaler = StandardScaler()
X_feats_train_scaled = scaler.fit_transform(X_feats_train)
X_feats_test_scaled  = scaler.transform(X_feats_test)
df_train_scaled = pd.DataFrame(X_feats_train_scaled, columns=feature_names)
df_test_scaled  = pd.DataFrame(X_feats_test_scaled,  columns=feature_names)

# --- PLOT 1: Train - Before Scaling (Log Scale) ---
plt.figure(figsize=(15, 5))
sns.boxplot(data=df_train_raw, palette="pastel")
plt.yscale('log')
plt.title("Train - Handcrafted Features Before Standardization (Log Scale)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# --- PLOT 2: Train - After Scaling ---
plt.figure(figsize=(15, 5))
sns.boxplot(data=df_train_scaled, palette="husl")
plt.title("Train - Handcrafted Features After Standardization")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# --- PLOT 3: Test - Before Scaling (Log Scale) ---
plt.figure(figsize=(15, 5))
sns.boxplot(data=df_test_raw, palette="pastel")
plt.yscale('log')
plt.title("Test - Handcrafted Features Before Standardization (Log Scale)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# --- PLOT 4: Test - After Scaling ---
plt.figure(figsize=(15, 5))
sns.boxplot(data=df_test_scaled, palette="husl")
plt.title("Test - Handcrafted Features After Standardization")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# --- MOD-SE(2) CNN (UNCHANGED) ---
def z2_se2n(input_tensor, kernel, orientations_nb, periodicity=2 * np.pi, diskMask=True, padding='VALID'):
    print("Base Kernel:\n", kernel.numpy())
    kernel_stack = rotate_lifting_kernels(kernel, orientations_nb, periodicity=periodicity, diskMask=diskMask)
    print("Z2-SE2N ROTATED KERNEL SET SHAPE:", kernel_stack.get_shape())
    kernels_as_if_2D = tf.transpose(kernel_stack, [1, 2, 3, 0, 4])
    kernelSizeH, kernelSizeW, channelsIN, channelsOUT = map(int, kernel.shape)
    kernels_as_if_2D = tf.reshape(kernels_as_if_2D, [kernelSizeH, kernelSizeW, channelsIN, orientations_nb * channelsOUT])
    layer_output = tf.nn.conv2d(input=input_tensor, filters=kernels_as_if_2D, strides=[1, 1, 1, 1], padding=padding)
    layer_output = tf.reshape(layer_output, [tf.shape(layer_output)[0], int(layer_output.shape[1]), int(layer_output.shape[2]), orientations_nb, channelsOUT])
    print("OUTPUT SE2N ACTIVATIONS SHAPE:", layer_output.get_shape())
    return layer_output, kernel_stack

def se2n_se2n(input_tensor, kernel, periodicity=2 * np.pi, diskMask=True, padding='VALID'):
    kernelSizeH, kernelSizeW, orientations_nb, channelsIN, channelsOUT = map(int, kernel.shape)
    kernel_stack = rotate_gconv_kernels(kernel, periodicity, diskMask)
    print("SE2N-SE2N ROTATED KERNEL SET SHAPE:", kernel_stack.get_shape())
    input_tensor_as_if_2D = tf.reshape(input_tensor, [tf.shape(input_tensor)[0], int(input_tensor.shape[1]), int(input_tensor.shape[2]), orientations_nb * channelsIN])
    kernels_as_if_2D = tf.transpose(kernel_stack, [1, 2, 3, 4, 0, 5])
    kernels_as_if_2D = tf.reshape(kernels_as_if_2D, [kernelSizeH, kernelSizeW, orientations_nb * channelsIN, orientations_nb * channelsOUT])
    layer_output = tf.nn.conv2d(input=input_tensor_as_if_2D, filters=kernels_as_if_2D, strides=[1, 1, 1, 1], padding=padding)
    layer_output = tf.reshape(layer_output, [tf.shape(layer_output)[0], int(layer_output.shape[1]), int(layer_output.shape[2]), orientations_nb, channelsOUT])
    print("OUTPUT SE2N ACTIVATIONS SHAPE:", layer_output.get_shape())
    return layer_output, kernel_stack

def spatial_max_pool(input_tensor, nbOrientations, padding='SAME'):
    activations = [None] * nbOrientations
    for i in range(nbOrientations):
        activations[i] = tf.nn.max_pool(value=input_tensor[:, :, :, i, :], ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding=padding)
    tensor_pooled = tf.stack(activations, axis=-1)
    return tensor_pooled

def rotate_lifting_kernels(kernel, orientations_nb, periodicity=2 * np.pi, diskMask=True):
    kernelSizeH, kernelSizeW, channelsIN, channelsOUT = map(int, kernel.shape)
    print("Z2-SE2N BASE KERNEL SHAPE:", kernel.get_shape())
    kernel_flat = tf.reshape(kernel, [kernelSizeH * kernelSizeW, channelsIN * channelsOUT])
    idx, vals = MultiRotationOperatorMatrixSparse([kernelSizeH, kernelSizeW], orientations_nb, periodicity=periodicity, diskMask=diskMask)
    rotOp_matrix = tf.SparseTensor(idx, vals, [orientations_nb * kernelSizeH * kernelSizeW, kernelSizeH * kernelSizeW])
    set_of_rotated_kernels = tf.sparse_tensor_dense_matmul(rotOp_matrix, kernel_flat)
    set_of_rotated_kernels = tf.reshape(set_of_rotated_kernels, [orientations_nb, kernelSizeH, kernelSizeW, channelsIN, channelsOUT])
    return set_of_rotated_kernels

def rotate_gconv_kernels(kernel, periodicity=2 * np.pi, diskMask=True):
    kernelSizeH, kernelSizeW, orientations_nb, channelsIN, channelsOUT = map(int, kernel.shape)
    print("SE2N-SE2N BASE KERNEL SHAPE:", kernel.get_shape())
    kernel_flat = tf.reshape(kernel, [kernelSizeH * kernelSizeW, orientations_nb * channelsIN * channelsOUT])
    idx, vals = MultiRotationOperatorMatrixSparse([kernelSizeH, kernelSizeW], orientations_nb, periodicity=periodicity, diskMask=diskMask)
    rotOp_matrix = tf.SparseTensor(idx, vals, [orientations_nb * kernelSizeH * kernelSizeW, kernelSizeH * kernelSizeW])
    kernels_planar_rotated = tf.sparse_tensor_dense_matmul(rotOp_matrix, kernel_flat)
    kernels_planar_rotated = tf.reshape(kernels_planar_rotated, [orientations_nb, kernelSizeH, kernelSizeW, orientations_nb, channelsIN, channelsOUT])
    set_of_rotated_kernels = [None] * orientations_nb
    for orientation in range(orientations_nb):
        kernels_temp = kernels_planar_rotated[orientation]
        kernels_temp = tf.transpose(kernels_temp, [0, 1, 3, 4, 2])
        kernels_temp = tf.reshape(kernels_temp, [kernelSizeH * kernelSizeW * channelsIN * channelsOUT, orientations_nb])
        roll_matrix = tf.constant(np.roll(np.identity(orientations_nb), orientation, axis=1), dtype=tf.float32)
        kernels_temp = tf.matmul(kernels_temp, roll_matrix)
        kernels_temp = tf.reshape(kernels_temp, [kernelSizeH, kernelSizeW, channelsIN, channelsOUT, orientations_nb])
        kernels_temp = tf.transpose(kernels_temp, [0, 1, 4, 2, 3])
        set_of_rotated_kernels[orientation] = kernels_temp
    return tf.stack(set_of_rotated_kernels)

def CoordRotationInv(ij, NiNj, theta):
    centeri = np.floor(NiNj[0] / 2)
    centerj = np.floor(NiNj[1] / 2)
    ijOld = np.zeros([2])
    ijOld[0] = np.cos(theta) * (ij[0] - centeri) + np.sin(theta) * (ij[1] - centerj) + centeri
    ijOld[1] = -np.sin(theta) * (ij[0] - centeri) + np.cos(theta) * (ij[1] - centerj) + centerj
    return ijOld

def LinIntIndicesAndWeights(ij, NiNj):
    i, j = ij
    Ni, Nj = NiNj
    i1 = int(np.floor(i))
    i2 = i1 + 1
    j1 = int(np.floor(j))
    j2 = j1 + 1
    ti = i - i1
    tj = j - j1
    w11 = (1 - ti) * (1 - tj)
    w12 = (1 - ti) * tj
    w21 = ti * (1 - tj)
    w22 = ti * tj
    indicesAndWeights = []
    if (0 <= i1 < Ni) and (0 <= j1 < Nj):
        indicesAndWeights.append([i1, j1, w11])
    if (0 <= i1 < Ni) and (0 <= j2 < Nj):
        indicesAndWeights.append([i1, j2, w12])
    if (0 <= i2 < Ni) and (0 <= j1 < Nj):
        indicesAndWeights.append([i2, j1, w21])
    if (0 <= i2 < Ni) and (0 <= j2 < Nj):
        indicesAndWeights.append([i2, j2, w22])
    return indicesAndWeights

def ToLinearIndex(ij, NiNj):
    return ij[0] * NiNj[0] + ij[1]

def RotationOperatorMatrix(NiNj, theta, diskMask=True):
    Ni, Nj = NiNj
    cij = np.floor(Ni / 2)
    rotationMatrix = np.zeros([Ni * Nj, Ni * Nj])
    for i in range(NiNj[0]):
        for j in range(NiNj[0]):
            if not(diskMask) or ((i - cij) * (i - cij) + (j - cij) * (j - cij) <= (cij + 0.5) * (cij + 0.5)):
                linij = ToLinearIndex([i, j], NiNj)
                ijOld = CoordRotationInv([i, j], NiNj, theta)
                linIntIndicesAndWeights = LinIntIndicesAndWeights(ijOld, NiNj)
                for indexAndWeight in linIntIndicesAndWeights:
                    indexOld = [indexAndWeight[0], indexAndWeight[1]]
                    linIndexOld = ToLinearIndex(indexOld, NiNj)
                    weight = indexAndWeight[2]
                    rotationMatrix[linij, linIndexOld] = weight
    return rotationMatrix

def MultiRotationOperatorMatrixSparse(NiNj, Ntheta, periodicity=2 * np.pi, diskMask=True):
    idx, vals = [], []
    for r in range(Ntheta):
        idxr, valsr = RotationOperatorMatrixSparse(NiNj, periodicity * r / Ntheta, linIndOffset=r * NiNj[0] * NiNj[1], diskMask=diskMask)
        idx += idxr
        vals += valsr
    return idx, vals

def RotationOperatorMatrixSparse(NiNj, theta, diskMask=True, linIndOffset=0):
    Ni, Nj = NiNj
    cij = np.floor(Ni / 2)
    idx, vals = [], []
    for i in range(NiNj[0]):
        for j in range(NiNj[0]):
            if not(diskMask) or ((i - cij) * (i - cij) + (j - cij) * (j - cij) <= (cij + 0.5) * (cij + 0.5)):
                linij = ToLinearIndex([i, j], NiNj)
                ijOld = CoordRotationInv([i, j], NiNj, theta)
                linIntIndicesAndWeights = LinIntIndicesAndWeights(ijOld, NiNj)
                for indexAndWeight in linIntIndicesAndWeights:
                    indexOld = [indexAndWeight[0], indexAndWeight[1]]
                    linIndexOld = ToLinearIndex(indexOld, NiNj)
                    weight = indexAndWeight[2]
                    idx.append((linij + linIndOffset, linIndexOld))
                    vals.append(weight)
    return tuple(idx), tuple(vals)

def GroupConv2D(filters, kernel_size, strides=(1, 1), padding='same', groups=3):
    def layer(x):
        group_list = []
        in_channels = x.shape[-1]
        assert in_channels % groups == 0, f"Number of input channels ({in_channels}) must be divisible by groups ({groups})"
        group_size = in_channels // groups
        for i in range(groups):
            x_group = x[:, :, :, i * group_size : (i + 1) * group_size]
            group_conv = tf.keras.layers.Conv2D(filters // groups, kernel_size, strides=strides, padding=padding)(x_group)
            group_list.append(group_conv)
        x = Concatenate()(group_list)
        x = BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        return x
    return layer

def SE2MaxPooling2D(pool_size=(2, 2)):
    def layer(x):
        x = tf.keras.layers.MaxPooling2D(pool_size=pool_size)(x)
        return x
    return layer

def SE2LiftingLayer(x):
    N, H, W, C = x.shape
    assert C % 3 == 0, "Number of input channels must be divisible by 3"
    group_size = C // 3
    x = tf.keras.layers.Reshape((H, W, 3, group_size))(x)
    x = tf.keras.layers.Permute((1, 2, 4, 3))(x)
    return x

def create_SE2CNN_model(input_shape, num_classes, dropout_rate=0.5):
    input_layer = Input(shape=input_shape)
    x = input_layer
    x = GroupConv2D(32, (3, 3))(x)
    x = SE2MaxPooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = GroupConv2D(64, (3, 3))(x)
    x = SE2MaxPooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = GroupConv2D(128, (3, 3))(x)
    x = SE2MaxPooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = GroupConv2D(256, (3, 3))(x)
    x = SE2MaxPooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = GroupConv2D(512, (3, 3))(x)
    x = SE2MaxPooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = GroupConv2D(1024, (3, 3))(x)
    x = SE2MaxPooling2D()(x)
    x = Dropout(dropout_rate)(x)
    x = SE2LiftingLayer(x)
    x = tf.keras.layers.Flatten()(x)
    x = Dense(1056, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    output = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=input_layer, outputs=output)
    return model

In [ ]:
from tensorflow.keras.layers import Lambda

# --- BUILD HYBRID MODEL ---
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from transformers import TFViTModel

def build_hybrid_model(image_input_shape, feat_input_shape, umap_feat_shape, num_classes, dropout_rate=0.4):
    # --- Mod-SE(2) CNN Branch (Unchanged) ---
    image_input_se2 = Input(shape=image_input_shape, name='image_input_se2')
    cnn_branch = create_SE2CNN_model(image_input_shape, num_classes, dropout_rate)
    x_se2 = cnn_branch(image_input_se2)
    x_se2 = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(x_se2)
    x_se2 = BatchNormalization()(x_se2)
    x_se2 = Dropout(dropout_rate)(x_se2)

    # --- Handcrafted Feature Branch ---
    feat_input = Input(shape=feat_input_shape, name='feat_input')
    x_feat = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(feat_input)
    x_feat = BatchNormalization()(x_feat)
    x_feat = Dropout(dropout_rate)(x_feat)

    # --- UMAP Feature Branch ---
    umap_input = Input(shape=umap_feat_shape, name='umap_feat_input')
    x_umap = Dense(32, activation='relu', kernel_regularizer=l2(0.01))(umap_input)
    x_umap = BatchNormalization()(x_umap)
    x_umap = Dropout(dropout_rate)(x_umap)

    # --- Fusion ---
    combined = Concatenate()([x_se2, x_feat, x_umap])
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(combined)
    x = Dropout(dropout_rate)(x)
    output = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=[image_input_se2, feat_input, umap_input], outputs=output)
    return model

In [ ]:
import tensorflow as tf

def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-8, 1.0)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.math.pow(1 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * cross_entropy, axis=1))
    return loss

In [ ]:
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ✅ Use balanced integer labels from SMOTE
y_train_labels = y_train_bal  # already integer-encoded

# ✅ Compute class weights for focal loss / training balance
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_labels), y=y_train_labels)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}

# ✅ Define training callbacks
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1, mode='max'),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=10, verbose=1, mode='max')
]

# ✅ Build hybrid model (Mod-SE(2) + handcrafted + UMAP)
model_hybrid = build_hybrid_model(
    image_input_shape=(224, 224, 3),
    feat_input_shape=(20,),       # handcrafted
    umap_feat_shape=(2,),         # UMAP projection
    num_classes=4,
    dropout_rate=0.4
)

# ✅ Compile model with focal loss
model_hybrid.compile(
    optimizer=Adam(1e-5),
    loss=focal_loss(gamma=2.5, alpha=0.25),
    metrics=['accuracy']
)

# ✅ Optional: Use data augmentation on training images (not required if inputs are balanced already)
datagen = ImageDataGenerator(
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rotation_range=20,
    brightness_range=[0.8, 1.2],
    shear_range=0.2,
    fill_mode='nearest'
)
datagen.fit(X_img_train_bal)

# ✅ Set training and validation inputs
train_inputs = [X_img_train_bal, X_feat_train_bal, X_train_umap]
val_inputs = [X_img_test, X_feat_test_scaled, X_test_umap]

# ✅ Fit model
history = model_hybrid.fit(
    train_inputs, y_train_cat_bal,
    validation_data=(val_inputs, y_test_cat),
    batch_size=16,
    epochs=20,
    class_weight=class_weight_dict,
    #callbacks=callbacks,
    verbose=1
)


In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'], label='train_accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.legend()
plt.show()
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.show()

In [ ]:
from tensorflow.keras.utils import plot_model
plot_model(model_hybrid, to_file="model_3branch.architecture.png", show_shapes=True, show_layer_names=True)
model_hybrid.save("/home/D13K48009/Clara/Result/CrossDataset/Training12Testing3/TryFindingBestModel.h5")

In [ ]:
# --- PARAMETER TUNING FOR DECISION TREE ---
max_depth_range = [3, 5, 7, 9, 11]
min_samples_leaf_range = [1, 3, 5, 10, 15]
threshold_range = np.linspace(0.3, 0.9, 13)

y_true = np.argmax(y_test_cat, axis=1)
y_pred_proba = model_hybrid.predict([X_img_test, X_feat_test_scaled, X_test_umap], verbose=0)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)

best_acc = 0
best_params = {}

for max_depth, min_samples_leaf in tqdm.tqdm(itertools.product(max_depth_range, min_samples_leaf_range)):
    tree = DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=min_samples_leaf, random_state=42)
    tree.fit(X_train_umap, y_train_labels)

    def generate_rule_function(tree):
        tree_ = tree.tree_
        def recurse(node):
            if tree_.feature[node] != _tree.TREE_UNDEFINED:
                idx = tree_.feature[node]
                thr = tree_.threshold[node]
                return f"(x[{idx}] <= {thr}) and ({recurse(tree_.children_left[node])}) or " \
                       f"(x[{idx}] > {thr}) and ({recurse(tree_.children_right[node])})"
            else:
                pred = np.argmax(tree_.value[node][0])
                return f"(return_val := {pred})"
        func_code = f"def rule_based_predict(x):\n    global return_val\n    {recurse(0)}\n    return return_val"
        return func_code

    exec(generate_rule_function(tree), globals())
    y_pred_rule = np.array([rule_based_predict(x) for x in X_test_umap])

    for threshold in threshold_range:
        y_pred_fused = np.where(np.max(y_pred_proba, axis=1) < threshold, y_pred_rule, y_pred_hybrid)
        acc = accuracy_score(y_true, y_pred_fused)
        if acc > best_acc:
            best_acc = acc
            best_params = {
                'max_depth': max_depth,
                'min_samples_leaf': min_samples_leaf,
                'threshold': threshold,
                'tree_model': tree,
                'final_prediction': y_pred_fused.copy()
            }

print(f"✅ Best Accuracy: {best_acc:.4f}")
print("Best Parameters:", {k: v for k, v in best_params.items() if k != 'final_prediction'})

In [ ]:
import joblib
import os
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# --- SAVE DIRECTORY ---
SAVE_DIR = "/home/D13K48009/Clara/Result/CrossDataset/Training12Testing3/"
os.makedirs(SAVE_DIR, exist_ok=True)

# --- FINAL DECISION TREE BASED ON BEST PARAMS ---
tree_umap = DecisionTreeClassifier(
    max_depth=best_params['max_depth'],
    min_samples_leaf=best_params['min_samples_leaf'],
    random_state=42
)
tree_umap.fit(X_train_umap, y_train_labels)

# --- SAVE MODELS ---
joblib.dump(tree_umap, os.path.join(SAVE_DIR, "rule_tree_mixed.pkl"))
joblib.dump(umap_reducer, os.path.join(SAVE_DIR, "umap_model_mixed.pkl"))

# --- PLOT TREE ---
fig_width = 40 if tree_umap.get_depth() < 8 else 100
plt.figure(figsize=(fig_width, 20))
plot_tree(
    tree_umap,
    feature_names=[f"UMAP{i}" for i in range(X_train_umap.shape[1])],
    class_names=[f'MES {i}' for i in range(4)],
    filled=True,
    rounded=True,
    fontsize=12
)
plt.title(f"Best Decision Tree on UMAP Features\n(max_depth={best_params['max_depth']}, min_samples_leaf={best_params['min_samples_leaf']})")
plt.tight_layout()

# --- SAVE TREE IMAGE ---
tree_plot_path = os.path.join(SAVE_DIR, "tree_umap_visualization.png")
plt.savefig(tree_plot_path, dpi=300)
plt.show()

print(f"✅ Tree and UMAP model saved to: {SAVE_DIR}")
print(f"🖼️  Tree visualization saved as: {tree_plot_path}")


In [ ]:
def generate_rule_function(tree, feature_names=["UMAP0", "UMAP1"]):
    tree_ = tree.tree_
    feature_name = [feature_names[i] if i != _tree.TREE_UNDEFINED else "undefined!" for i in tree_.feature]
    def recurse(node, depth):
        indent = "    " * depth
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name = feature_name[node]
            threshold = tree_.threshold[node]
            return (
                f"{indent}if x[{feature_names.index(name)}] <= {threshold:.6f}:\n" +
                recurse(tree_.children_left[node], depth + 1) +
                f"{indent}else:\n" +
                recurse(tree_.children_right[node], depth + 1)
            )
        else:
            pred_class = np.argmax(tree_.value[node][0])
            return f"{indent}return {pred_class}  # MES {pred_class}\n"
    function_code = "def rule_based_predict_best(x):\n" + recurse(0, 1)
    return function_code

rule_function_code = generate_rule_function(tree_umap)
print(rule_function_code)
exec(rule_function_code, globals())

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import os

# --- Set Save Path ---
SAVE_DIR = "/home/D13K48009/Clara/Result/CrossDataset/Training12Testing3/"
os.makedirs(SAVE_DIR, exist_ok=True)
REPORT_PATH = os.path.join(SAVE_DIR, "evaluation_report.txt")

# --- Final Predictions ---
threshold = best_params['threshold']
y_pred_rule_umap = np.array([rule_based_predict_best(row) for row in X_test_umap])
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)
y_pred_combined = np.where(np.max(y_pred_proba, axis=1) < threshold, y_pred_rule_umap, y_pred_hybrid)
y_pred_override = np.array([hybrid_class if rule_class != hybrid_class else rule_class
                            for rule_class, hybrid_class in zip(y_pred_rule_umap, y_pred_hybrid)])
y_true = np.argmax(y_test_cat, axis=1)

# --- Generate Reports ---
report_hybrid = classification_report(y_true, y_pred_hybrid, digits=4)
report_combined = classification_report(y_true, y_pred_combined, digits=4)
report_override = classification_report(y_true, y_pred_override, digits=4)
report_rule = classification_report(y_true, y_pred_rule_umap, digits=4)

# --- Print to Console ---
print("📊 Hybrid Only:\n", report_hybrid)
print("📊 Rule-Aware Hybrid (Confidence-Guided Fallback):\n", report_combined)
print("📊 Rule-Aware Hybrid (Override When Agree):\n", report_override)
print("📊 Rule Only:\n", report_rule)

# --- Save to TXT File ---
with open(REPORT_PATH, 'w') as f:
    f.write("📊 Hybrid Only:\n")
    f.write(report_hybrid + "\n\n")
    f.write("📊 Rule-Aware Hybrid (Confidence-Guided Fallback):\n")
    f.write(report_combined + "\n\n")
    f.write("📊 Rule-Aware Hybrid (Override When Agree):\n")
    f.write(report_override + "\n\n")
    f.write("📊 Rule Only:\n")
    f.write(report_rule + "\n")

print(f"\n✅ Evaluation report saved to: {REPORT_PATH}")


In [ ]:
from flask import Flask, request, jsonify
import openai
import threading

openai.api_key = "YOUR_OPENAI_API_KEY_HERE"  # or from environment

app = Flask(__name__)

@app.route("/predict", methods=["POST"])
def predict():
    prompt = request.json.get("prompt", "")
    try:
        res = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=10
        )
        content = res['choices'][0]['message']['content']
        # Try to find valid class (0–3)
        for token in content.split():
            if token.isdigit() and int(token) in [0, 1, 2, 3]:
                return jsonify({"class": int(token), "raw": content})
        return jsonify({"class": None, "raw": content})
    except Exception as e:
        return jsonify({"error": str(e)})

# ✅ Run in separate thread
def run_flask():
    app.run(host='0.0.0.0', port=5000)

threading.Thread(target=run_flask).start()



In [ ]:
import os
import json
import numpy as np
from sklearn.metrics import classification_report

# --- Ensure UMAP projection is applied to test handcrafted features ---
X_feat_test_umap = umap_reducer.transform(X_feat_test_scaled)
y_true = np.argmax(y_test_cat, axis=1)  # <-- FIXED

# --- Predict using hybrid model ---
y_pred_proba = model_hybrid.predict(
    [X_img_test, X_feat_test_scaled, X_feat_test_umap], verbose=0
)
y_pred_hybrid = np.argmax(y_pred_proba, axis=1)
confidences = np.max(y_pred_proba, axis=1)

# --- Rule-based prediction ---
y_pred_rule_umap = np.array([rule_based_predict_best(row) for row in X_feat_test_umap])

# --- Fusion logic ---
threshold = best_params.get("threshold", 0.55)
y_pred_combined = np.where(confidences < threshold, y_pred_rule_umap, y_pred_hybrid)
y_pred_agree = np.array([
    rule if rule == model else rule
    for rule, model in zip(y_pred_rule_umap, y_pred_hybrid)
])

# --- Print evaluation ---
print("Hybrid Only:\n", classification_report(y_true, y_pred_hybrid, digits=4))
print("Rule-Aware Hybrid (Confidence-Guided):\n", classification_report(y_true, y_pred_combined, digits=4))
print("Rule-Aware Hybrid (Agreement Override):\n", classification_report(y_true, y_pred_agree, digits=4))
print("Rule Only:\n", classification_report(y_true, y_pred_rule_umap, digits=4))

# --- Save evaluation report to file ---
output_dir = "/home/D13K48009/Clara/Result/CrossDataset/Training12Testing3/"
os.makedirs(output_dir, exist_ok=True)

report_file = os.path.join(output_dir, "classification_report.txt")
with open(report_file, "w") as f:
    f.write("Hybrid Only:\n")
    f.write(classification_report(y_true, y_pred_hybrid, digits=4))
    f.write("\nRule-Aware Hybrid (Confidence-Guided):\n")
    f.write(classification_report(y_true, y_pred_combined, digits=4))
    f.write("\nRule-Aware Hybrid (Agreement Override):\n")
    f.write(classification_report(y_true, y_pred_agree, digits=4))
    f.write("\nRule Only:\n")
    f.write(classification_report(y_true, y_pred_rule_umap, digits=4))

print(f"✅ Classification report saved to: {report_file}")

# --- Save JSONL log for LLM feedback learning ---
log_file_path = os.path.join(output_dir, "llm_feedback_log_mixed.jsonl")
with open(log_file_path, "w") as f:
    for i in range(len(y_true)):
        entry = {
            "model_prediction": int(y_pred_hybrid[i]),
            "rule_prediction": int(y_pred_rule_umap[i]),
            "final_fusion_prediction": int(y_pred_combined[i]),
            "true_label": int(y_true[i]),
            "confidence": float(confidences[i]),
            "umap_0": float(X_feat_test_umap[i][0]),
            "umap_1": float(X_feat_test_umap[i][1]),
            "features": X_feat_test_scaled[i].tolist(),
            "feedback": ""
        }

        # Label feedback
        if y_pred_combined[i] != y_true[i]:
            if confidences[i] < threshold and y_pred_rule_umap[i] == y_true[i]:
                entry["feedback"] = "Should have used the rule-based prediction. Confidence was low."
            elif confidences[i] >= threshold and y_pred_hybrid[i] == y_true[i]:
                entry["feedback"] = "Correctly used the model prediction. Confidence was high."
            else:
                entry["feedback"] = "Incorrect prediction. Consider learning from UMAP and features."
        else:
            entry["feedback"] = "Correct prediction."

        f.write(json.dumps(entry) + "\n")

print(f"✅ Logged {len(y_true)} entries to {log_file_path}")


In [ ]:
import openai
import json
import time
from tqdm import tqdm

# ✅ Set your OpenAI API Key
openai.api_key = "YOUR_OPENAI_API_KEY_HERE"  # Replace with your OpenAI key

# ✅ Path to your saved .jsonl log
log_file_path = "llm_feedback_log_mixed.jsonl"

# ✅ Load prediction entries
with open(log_file_path, "r") as f:
    entries = [json.loads(line) for line in f]

# ✅ Function to build prompt from entry
def build_prompt(entry):
    return f"""You are a colonoscopy diagnosis assistant trained to override model predictions when necessary.

Model prediction: {entry['model_prediction']}
Rule prediction: {entry['rule_prediction']}
Confidence: {entry['confidence']:.2f}
UMAP: ({entry['umap_0']:.2f}, {entry['umap_1']:.2f})
Handcrafted features: {entry['features']}

True label: {entry['true_label']}

Task: Given all the information, what should the final predicted MES class be?
Only respond with a single number: 0, 1, 2, or 3.
"""

# ✅ Function to get LLM recommendation
def query_gpt(prompt, model="gpt-3.5-turbo", retries=3):
    for attempt in range(retries):
        try:
            response = openai.ChatCompletion.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            return response.choices[0].message["content"].strip()
        except Exception as e:
            print(f"Error: {e}. Retrying...")
            time.sleep(1)
    return None

# ✅ Run through all entries and get LLM-based decisions
llm_preds = []
for entry in tqdm(entries, desc="LLM predicting"):
    prompt = build_prompt(entry)
    prediction = query_gpt(prompt)
    try:
        pred_int = int(prediction)
        if pred_int in [0, 1, 2, 3]:
            llm_preds.append(pred_int)
        else:
            llm_preds.append(entry["final_fusion_prediction"])  # fallback
    except:
        llm_preds.append(entry["final_fusion_prediction"])  # fallback

# ✅ Calculate accuracy of LLM override agent
true_labels = [e["true_label"] for e in entries]
correct = sum([int(p == t) for p, t in zip(llm_preds, true_labels)])
accuracy = correct / len(true_labels)

print(f"✅ LLM Agent Accuracy: {accuracy:.4f}")


In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from sklearn.metrics import accuracy_score, classification_report

# --- Setup ---
output_dir = "/home/D13K48009/Clara/Result/CrossDataset/Training12Testing3/"
os.makedirs(output_dir, exist_ok=True)

# ✅ Load previous log
log_file_path = os.path.join(output_dir, "llm_feedback_log_mixed.jsonl")
with open(log_file_path, "r") as f:
    entries = [json.loads(line) for line in f]

# ✅ Update and overwrite JSONL with LLM predictions
updated_log_path = os.path.join(output_dir, "llm_feedback_log_updated_mixed.jsonl")
with open(updated_log_path, "w") as f:
    for entry, llm_pred in zip(entries, llm_preds):
        entry["llm_prediction"] = llm_pred
        entry["llm_correct"] = int(llm_pred == entry["true_label"])
        f.write(json.dumps(entry) + "\n")

print(f"✅ Updated JSONL written to: {updated_log_path}")

# ✅ Collect predictions and labels
fusion_preds = [e["final_fusion_prediction"] for e in entries]
rule_preds = [e["rule_prediction"] for e in entries]
model_preds = [e["model_prediction"] for e in entries]
llm_preds = [e.get("llm_prediction", e["final_fusion_prediction"]) for e in entries]
true_labels = [e["true_label"] for e in entries]

# ✅ Per-class accuracy
methods = {
    "Model": model_preds,
    "Rule": rule_preds,
    "Fusion": fusion_preds,
    "LLM": llm_preds
}
class_labels = [0, 1, 2, 3]
acc_by_class = defaultdict(dict)

for method, preds in methods.items():
    for cls in class_labels:
        cls_indices = [i for i, t in enumerate(true_labels) if t == cls]
        cls_correct = sum([preds[i] == true_labels[i] for i in cls_indices])
        acc_by_class[method][cls] = cls_correct / len(cls_indices) if cls_indices else 0

# ✅ Bar Plot per-class accuracy
x = np.arange(len(class_labels))
width = 0.2
plt.figure(figsize=(10, 6))
for i, (method, accs) in enumerate(acc_by_class.items()):
    plt.bar(x + i*width, [accs[cls] for cls in class_labels], width=width, label=method)

plt.xticks(x + 1.5*width, [f"MES {cls}" for cls in class_labels])
plt.ylabel("Accuracy")
plt.title("Accuracy by MES Class per Method")
plt.legend()
plt.grid(True, axis='y')
plt.tight_layout()

# ✅ Save plot
plot_path = os.path.join(output_dir, "accuracy_by_class_per_method.png")
plt.savefig(plot_path)
plt.show()
print(f"✅ Saved accuracy plot to: {plot_path}")

# ✅ Print and save classification reports
def print_and_save_report(name, preds):
    acc = accuracy_score(true_labels, preds)
    report = classification_report(true_labels, preds, target_names=[f"MES {i}" for i in range(4)], digits=4)
    print(f"\n🔍 {name} Accuracy: {acc:.4f}")
    print(report)

    report_path = os.path.join(output_dir, f"{name.lower().replace(' ', '_')}_report.txt")
    with open(report_path, "w") as f:
        f.write(f"{name} Accuracy: {acc:.4f}\n\n")
        f.write(report)
    print(f"✅ Saved report to: {report_path}")

print_and_save_report("Model Only", model_preds)
print_and_save_report("Rule-Based", rule_preds)
print_and_save_report("Fusion", fusion_preds)
print_and_save_report("LLM Override", llm_preds)


In [ ]:
# === STEP 1: Model Prediction on Training Set ===
# y_pred_proba_train: softmax probabilities
import json
import os

def create_feedback_jsonl(
    filename,
    y_true,
    y_model_pred,
    y_rule_pred,
    proba,
    umap_feat,
    handcrafted_feat,
    save_path="/home/D13K48009/Clara/Result/CrossDataset/Training12Testing3/"
):
    assert len(y_true) == len(y_model_pred) == len(y_rule_pred) == len(proba) == len(umap_feat) == len(handcrafted_feat)
    records = []
    for i in range(len(y_true)):
        record = {
            "true_label": int(y_true[i]),
            "model_pred": int(y_model_pred[i]),
            "rule_pred": int(y_rule_pred[i]),
            "confidence": float(np.max(proba[i])),
            "proba": list(map(float, proba[i])),
            "umap_0": float(umap_feat[i][0]),
            "umap_1": float(umap_feat[i][1]),
            "features": list(map(float, handcrafted_feat[i]))
        }
        records.append(record)
    
    full_path = os.path.join(save_path, filename)
    with open(full_path, "w") as f:
        for rec in records:
            f.write(json.dumps(rec) + "\n")
    
    print(f"✅ Saved {len(records)} samples to {full_path}")

# --- Predict on training set using all 3 branches ---
y_pred_proba_train = model_hybrid.predict(
    [X_img_train_bal, X_feat_train_bal, X_train_umap],
    verbose=1
)
y_pred_hybrid_train = np.argmax(y_pred_proba_train, axis=1)

# === STEP 2: Rule-Based Prediction on UMAP Train ===
y_pred_rule_umap_train = np.array([rule_based_predict_best(row) for row in X_train_umap])

# === STEP 3: Save feedback_train.jsonl ===
create_feedback_jsonl(
    filename="feedback_train.jsonl",
    y_true=y_train_bal,                       # SMOTE-balanced true labels
    y_model_pred=y_pred_hybrid_train,         # model prediction
    y_rule_pred=y_pred_rule_umap_train,       # rule-based prediction
    proba=y_pred_proba_train,                 # model probabilities
    umap_feat=X_train_umap,                   # 2D UMAP features
    handcrafted_feat=X_feat_train_bal      # original scaled 20D handcrafted features
)

In [ ]:
import os
import json
import numpy as np

# Define base path for saving
base_dir = "/home/D13K48009/Clara/Result/CrossDataset/Training12Testing3/"

def create_feedback_jsonl(filename, y_true, y_model_pred, y_rule_pred, proba, umap_feat, handcrafted_feat):
    records = []
    for i in range(len(y_true)):
        entry = {
            "true_label": int(y_true[i]),
            "model_prediction": int(y_model_pred[i]),
            "rule_prediction": int(y_rule_pred[i]),
            "confidence": float(np.max(proba[i])),
            "umap_0": float(umap_feat[i, 0]),
            "umap_1": float(umap_feat[i, 1]),
            "features": [float(f) for f in handcrafted_feat[i]]
        }
        records.append(entry)

    path = os.path.join(base_dir, filename)
    with open(path, "w") as f:
        for row in records:
            f.write(json.dumps(row) + "\n")
    print(f"✅ Saved {len(records)} samples to {path}")

# === GENERATE TESTING SET FEEDBACK FILE ===
create_feedback_jsonl(
    filename="feedback_test.jsonl",
    y_true=y_test_encoded,
    y_model_pred=y_pred_hybrid,
    y_rule_pred=y_pred_rule_umap,
    proba=y_pred_proba,
    umap_feat=X_feat_test_umap,
    handcrafted_feat=X_feat_test_scaled
)


In [ ]:
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import os
import joblib
import warnings
from hashlib import sha1

warnings.filterwarnings("ignore")

# === PATHS ===
train_file = "/home/D13K48009/Clara/Result/CrossDataset/Training12Testing3/feedback_train.jsonl"
test_file  = "/home/D13K48009/Clara/Result/CrossDataset/Training12Testing3/feedback_test.jsonl"
save_dir   = "/home/D13K48009/Clara/Result/CrossDataset/Training12Testing3/"
os.makedirs(save_dir, exist_ok=True)

# === LOAD FUNCTION ===
def load_feedback_jsonl(path):
    with open(path, "r") as f:
        data = [json.loads(line.strip()) for line in f]
    df = pd.DataFrame(data)
    df["label"] = df["true_label"]
    return df

df_train = load_feedback_jsonl(train_file)
df_test  = load_feedback_jsonl(test_file)
df_test_orig = df_test.copy()  # Freeze

# === FEATURE ENCODING ===
def encode_features(df):
    df_feat = df[["confidence", "umap_0", "umap_1"]].copy()
    for i in range(20):
        df_feat[f"f{i}"] = df["features"].apply(lambda x: x[i])
    return df_feat.values, df["label"].values

X_train, y_train = encode_features(df_train)
X_test, y_test   = encode_features(df_test_orig)

# === SCALING ===
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# === TRAIN LOOP ===
loop = 0
acc_list = []
known_misclassified = set()

while True:
    clf = lgb.LGBMClassifier(random_state=42, class_weight='balanced')
    clf.fit(X_train_scaled, y_train)

    y_pred = clf.predict(X_test_scaled)
    y_proba = clf.predict_proba(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    acc_list.append(acc)

    print(f"🔁 Loop {loop+1}: Accuracy = {acc:.4f}")
    if acc >= 0.90:
        print("✅ Target reached.")
        break

    misclassified = df_test_orig[y_pred != y_test].copy()
    misclassified["hash"] = misclassified.apply(
        lambda row: sha1(json.dumps(row.to_dict(), sort_keys=True).encode()).hexdigest(), axis=1
    )
    new_errors = misclassified[~misclassified["hash"].isin(known_misclassified)]

    if new_errors.empty:
        print("⚠️ No new unique misclassified samples to learn from.")
        break

    print(f"➕ Adding {len(new_errors)} new feedback samples")
    known_misclassified.update(new_errors["hash"])
    df_train = pd.concat([df_train, new_errors.drop(columns=["hash"])], ignore_index=True)

    X_train, y_train = encode_features(df_train)
    X_train_scaled = scaler.fit_transform(X_train)

    loop += 1

# === SAVE FINAL AGENT ===
clf.booster_.save_model(os.path.join(save_dir, "feedback_agent.txt"))
joblib.dump(scaler, os.path.join(save_dir, "scaler_agent.pkl"))

# === SAVE LEARNING CURVE ===
plt.figure(figsize=(10, 5))
plt.plot(acc_list, marker='o', label="Test Accuracy")
plt.axhline(0.90, color='red', linestyle='--', label="Target")
plt.title("Agent Continual Learning Curve")
plt.xlabel("Iteration")
plt.ylabel("Accuracy")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "agent_learning_curve.png"))

# === FINAL REPORT ===
report = classification_report(y_test, y_pred, digits=4, output_dict=True)
report_df = pd.DataFrame(report).T
report_df.to_csv(os.path.join(save_dir, "agent_final_classification_report.csv"))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import joblib
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize
import lightgbm as lgb

# === PATHS ===
save_dir = "/home/D13K48009/Clara/Result/CrossDataset/Training12Testing3/"
test_file = os.path.join(save_dir, "feedback_test.jsonl")
agent_file = os.path.join(save_dir, "feedback_agent.txt")
scaler_file = os.path.join(save_dir, "scaler_agent.pkl")

# === LOAD TEST SET ===
with open(test_file, "r") as f:
    test_data = [json.loads(line.strip()) for line in f]
df_test = pd.DataFrame(test_data)
df_test["label"] = df_test["true_label"]

# === ENCODE FEATURES ===
def encode_features(df):
    df_feat = df[["confidence", "umap_0", "umap_1"]].copy()
    for i in range(20):
        df_feat[f"f{i}"] = df["features"].apply(lambda x: x[i])
    return df_feat.values, df["label"].values

X_test, y_true = encode_features(df_test)
scaler = joblib.load(scaler_file)
X_test_scaled = scaler.transform(X_test)

# === LOAD AGENT & PREDICT ===
agent = lgb.Booster(model_file=agent_file)
y_proba = agent.predict(X_test_scaled)
y_pred = np.argmax(y_proba, axis=1)

# === METRICS ===
n_classes = 4
class_names = [f"MES {i}" for i in range(n_classes)]
y_true_oh = label_binarize(y_true, classes=list(range(n_classes)))
report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
df_report = pd.DataFrame(report).T

# === Per-Class Custom Metrics ===
cm = confusion_matrix(y_true, y_pred)
metrics_dict = {}
for i in range(n_classes):
    TP = cm[i, i]
    FN = np.sum(cm[i, :]) - TP
    FP = np.sum(cm[:, i]) - TP
    TN = np.sum(cm) - (TP + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    npv       = TN / (TN + FN) if (TN + FN) > 0 else 0
    ppv       = precision
    accuracy  = (TP + TN) / (TP + TN + FP + FN)
    metrics_dict[f"MES {i}"] = {
        "precision": precision,
        "recall": recall,
        "f1-score": f1,
        "npv": npv,
        "ppv": ppv,
        "accuracy": accuracy
    }

# === Overall Macro Average ===
all_vals = pd.DataFrame(metrics_dict).T
metrics_dict["Overall"] = {
    metric: all_vals[metric].mean() for metric in all_vals.columns
}
summary = pd.DataFrame(metrics_dict).T
summary.to_csv(os.path.join(save_dir, "agent_evaluation_summary.csv"))
print("📊 Evaluation Summary:\n", summary.round(4))

# === Radar Chart ===
metrics = ["precision", "recall", "f1-score", "npv"]
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

plt.figure(figsize=(8, 6))
ax = plt.subplot(111, polar=True)
for i in range(n_classes):
    label = f"MES {i}"
    values = summary.loc[label, metrics].values.tolist()
    values += values[:1]
    ax.plot(angles, values, label=label)
    ax.fill(angles, values, alpha=0.1)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
plt.xticks(angles[:-1], metrics)
plt.yticks([0.25, 0.5, 0.75, 1.0], ["0.25", "0.5", "0.75", "1.0"])
plt.title("Radar Chart: Per-Class Evaluation")
plt.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "agent_radar_chart.png"))
plt.show()


# === Confusion Matrix ===
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax[0],
            xticklabels=class_names, yticklabels=class_names)
ax[0].set_title("Confusion Matrix")
ax[0].set_xlabel("Predicted")
ax[0].set_ylabel("True")
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", ax=ax[1],
            xticklabels=class_names, yticklabels=class_names)
ax[1].set_title("Normalized Confusion Matrix")
ax[1].set_xlabel("Predicted")
ax[1].set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "agent_confusion_matrix.png"))
plt.show()

# === ROC Curve ===
fpr = dict(); tpr = dict(); roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_oh[:, i], y_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
fpr["micro"], tpr["micro"], _ = roc_curve(y_true_oh.ravel(), y_proba.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
roc_auc["macro"] = roc_auc_score(y_true_oh, y_proba, average="macro")

plt.figure(figsize=(8, 6))
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], label=f"MES {i} (AUC = {roc_auc[i]:.2f})")
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.title("ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "agent_roc_curve.png"))
plt.show()


In [ ]:
# === Radar Chart ===
metrics = ["accuracy", "precision", "recall", "f1-score"]
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]  # to close the radar chart loop

plt.figure(figsize=(8, 6))
ax = plt.subplot(111, polar=True)

for i in range(n_classes):
    label = f"MES {i}"
    values = summary.loc[label, metrics].values.tolist()
    values += values[:1]
    ax.plot(angles, values, label=label)
    ax.fill(angles, values, alpha=0.1)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
plt.xticks(angles[:-1], metrics)
plt.yticks([0.25, 0.5, 0.75, 1.0], ["0.25", "0.5", "0.75", "1.0"])
plt.title("Radar Chart: Accuracy, Precision, Recall, F1-Score (Per Class)")
plt.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "agent_radar_chart_accuracy.png"))
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# === Define save directory ===
save_dir = "/home/D13K48009/Clara/Result/CrossDataset/Training12Testing3/"

# === Handcrafted feature names: 20D (from Wavelet + GLCM) ===
handcrafted_feature_names = [
    "LL_Mean", "LL_Std", "LL_Var", "LL_Entropy",
    "LH_Mean", "LH_Std", "LH_Var", "LH_Entropy",
    "HL_Mean", "HL_Std", "HL_Var", "HL_Entropy",
    "HH_Mean", "HH_Std", "HH_Var", "HH_Entropy",
    "HH_Energy",
    "GLCM_Contrast", "GLCM_Dissimilarity", "GLCM_Homogeneity"
]

# === Get feature importances using LightGBM Booster method ===
importances = agent.feature_importance()

# === Handcrafted features are from index 3 to 22 (inclusive) ===
handcrafted_importances = importances[3:23]

# === Sort descending ===
sorted_idx = np.argsort(handcrafted_importances)[::-1]
sorted_names = [handcrafted_feature_names[i] for i in sorted_idx]
sorted_vals = handcrafted_importances[sorted_idx]

# === Plot bar chart ===
plt.figure(figsize=(10, 6))
plt.barh(sorted_names, sorted_vals, color='skyblue')
plt.gca().invert_yaxis()
plt.xlabel("Importance Score")
plt.title("Handcrafted Feature Importance (LightGBM Agent)")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "agent_handcrafted_feature_importance.png"))
plt.show()


In [ ]:
# Get percentage of each handcrafted feature's importance
importance_sum = handcrafted_importances.sum()
percentages = (handcrafted_importances / importance_sum) * 100

# Print as a table
for name, score, pct in zip(sorted_names, sorted_vals, percentages[sorted_idx]):
    print(f"{name:<20} | Score: {score:>4} | Percentage: {pct:>5.2f}%")

In [ ]:
split_importance = agent.feature_importance(importance_type='split')
gain_importance  = agent.feature_importance(importance_type='gain')

hand_split = split_importance[3:23]
hand_gain  = gain_importance[3:23]

split_pct = (hand_split / hand_split.sum()) * 100
gain_pct  = (hand_gain / hand_gain.sum()) * 100

for i, name in enumerate(handcrafted_feature_names):
    print(f"{name:<20} | Split: {hand_split[i]:>4} ({split_pct[i]:>5.2f}%) | Gain: {hand_gain[i]:>7.2f} ({gain_pct[i]:>5.2f}%)")


In [ ]:
import pandas as pd

# Get LightGBM importances
split_importance = agent.feature_importance(importance_type='split')
gain_importance  = agent.feature_importance(importance_type='gain')

# Define handcrafted features
start, end = 3, 23
handcrafted_feature_names = [
    "LL_Mean", "LL_Std", "LL_Var", "LL_Entropy",
    "LH_Mean", "LH_Std", "LH_Var", "LH_Entropy",
    "HL_Mean", "HL_Std", "HL_Var", "HL_Entropy",
    "HH_Mean", "HH_Std", "HH_Var", "HH_Entropy",
    "HH_Energy",
    "GLCM_Contrast", "GLCM_Dissimilarity", "GLCM_Homogeneity"
]

# Extract and normalize
hand_split = split_importance[start:end]
hand_gain  = gain_importance[start:end]
split_pct = (hand_split / hand_split.sum()) * 100
gain_pct  = (hand_gain / hand_gain.sum()) * 100

# Build dataframe
df = pd.DataFrame({
    'Feature': handcrafted_feature_names,
    'Split Score': hand_split,
    'Split %': split_pct,
    'Gain Score': hand_gain,
    'Gain %': gain_pct
})

# Top 5
top5_split = df.sort_values(by='Split Score', ascending=False).head(5).reset_index(drop=True)
top5_gain  = df.sort_values(by='Gain Score', ascending=False).head(5).reset_index(drop=True)

# Combine into side-by-side table
combined_df = pd.DataFrame({
    'Top 5 (Split)': top5_split['Feature'],
    'Split %': top5_split['Split %'].round(2),
    'Top 5 (Gain)': top5_gain['Feature'],
    'Gain %': top5_gain['Gain %'].round(2)
})

# Show in notebook
display(combined_df)


In [ ]:
%%writefile error_taxonomy.py
from typing import Optional, Sequence, Dict, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

ERROR_TYPES = [
    "both_correct",
    "fixed_by_agent",      # CNN wrong → Agent right
    "harmed_by_agent",     # CNN right → Agent wrong
    "both_wrong_same",     # both wrong, same wrong class
    "both_wrong_diff",     # both wrong, different wrong classes
]

def _to_numpy_int(a):
    a = np.asarray(a)
    if a.ndim != 1:
        raise ValueError("y arrays must be 1D")
    return a.astype(int)

def _max_conf(proba: Optional[np.ndarray]) -> Optional[np.ndarray]:
    if proba is None:
        return None
    p = np.asarray(proba)
    # LightGBM binary sometimes yields 1D; make it (n,2)
    if p.ndim == 1:
        p = np.vstack([1 - p, p]).T
    if p.ndim != 2:
        raise ValueError("proba must be 2D: shape (n_samples, n_classes)")
    return p.max(axis=1)

def _infer_class_names(n_classes: int, class_names: Optional[Sequence[str]]):
    if class_names is None:
        return [f"class_{i}" for i in range(n_classes)]
    if len(class_names) != n_classes:
        raise ValueError("class_names length must match number of classes")
    return list(class_names)

def _error_label(true_i, cnn_i, agent_i) -> str:
    cnn_ok = (cnn_i == true_i)
    agent_ok = (agent_i == true_i)
    if cnn_ok and agent_ok: return "both_correct"
    if (not cnn_ok) and agent_ok: return "fixed_by_agent"
    if cnn_ok and (not agent_ok): return "harmed_by_agent"
    if (not cnn_ok) and (not agent_ok):
        return "both_wrong_same" if (cnn_i == agent_i) else "both_wrong_diff"
    raise RuntimeError("Unreachable")

def build_error_taxonomy(
    y_true: Sequence[int],
    y_cnn: Sequence[int],
    y_agent: Sequence[int],
    cnn_proba: Optional[np.ndarray] = None,
    agent_proba: Optional[np.ndarray] = None,
    class_names: Optional[Sequence[str]] = None,
    low_conf_cutoff: float = 0.55,
) -> Dict[str, Any]:
    y_true = _to_numpy_int(y_true)
    y_cnn  = _to_numpy_int(y_cnn)
    y_agent= _to_numpy_int(y_agent)
    n = len(y_true)
    if not (len(y_cnn) == len(y_agent) == n):
        raise ValueError("All y arrays must have the same length")

    n_classes = int(np.max([y_true.max(), y_cnn.max(), y_agent.max()])) + 1
    names = _infer_class_names(n_classes, class_names)

    cnn_conf   = _max_conf(cnn_proba)
    agent_conf = _max_conf(agent_proba)

    err_labels = np.array([_error_label(t, c, a) for t, c, a in zip(y_true, y_cnn, y_agent)])
    override   = (y_cnn != y_agent)
    cnn_ok     = (y_cnn == y_true)
    agent_ok   = (y_agent == y_true)

    df = pd.DataFrame({
        "true": y_true,
        "cnn_pred": y_cnn,
        "agent_pred": y_agent,
        "cnn_correct": cnn_ok,
        "agent_correct": agent_ok,
        "override": override,
        "error_type": err_labels
    })

    if cnn_conf is not None:
        df["cnn_conf"] = cnn_conf
        df["cnn_low_conf"] = (cnn_conf < low_conf_cutoff)
    if agent_conf is not None:
        df["agent_conf"] = agent_conf
        df["agent_low_conf"] = (agent_conf < low_conf_cutoff)

    # Per-true-class taxonomy
    df["true_name"] = pd.Categorical(df["true"].map(lambda i: names[i]), categories=names, ordered=True)
    by_true_err = (
        df.pivot_table(index="true_name", columns="error_type", values="cnn_pred", aggfunc="count", fill_value=0)
        .reindex(columns=ERROR_TYPES, fill_value=0)
    )
    by_true_err_pct = by_true_err.div(by_true_err.sum(axis=1).replace(0, 1), axis=0) * 100.0

    # Overrides detail: true × "cnn→agent"
    trans = df[df["override"]].copy()
    if not trans.empty:
        trans["pair"] = trans.apply(lambda r: f'{names[r["cnn_pred"]]}→{names[r["agent_pred"]]}', axis=1)
        overrides_by_pair_cnt = (
            trans.pivot_table(index="true_name", columns="pair", values="override", aggfunc="count", fill_value=0)
            .sort_index(axis=1)
        )
    else:
        overrides_by_pair_cnt = pd.DataFrame(index=by_true_err.index)

    # McNemar components
    a = int(np.sum(cnn_ok & agent_ok))
    b = int(np.sum(cnn_ok & (~agent_ok)))
    c = int(np.sum((~cnn_ok) & agent_ok))
    d = int(np.sum((~cnn_ok) & (~agent_ok)))
    bc = b + c
    if bc == 0:
        p_value = 1.0
    else:
        chi = ((abs(b - c) - 1) ** 2) / bc
        p_value = np.exp(-chi / 2) * (1 + chi / 2)  # 1-dof χ² tail approx

    # Confusions
    cm_cnn   = confusion_matrix(y_true, y_cnn, labels=list(range(n_classes)))
    cm_agent = confusion_matrix(y_true, y_agent, labels=list(range(n_classes)))

    return {
        "taxonomy_df": df,
        "by_true_error_type_cnt": by_true_err,
        "by_true_error_type_pct": by_true_err_pct,
        "overrides_by_pair_cnt": overrides_by_pair_cnt,
        "mcnemar_table": {"a_both_correct": a, "b_cnn_only": b, "c_agent_only": c, "d_both_wrong": d, "p_value_approx": p_value},
        "confusions": {"cnn": cm_cnn, "agent": cm_agent},
        "class_names": names
    }

def plot_error_taxonomy_heatmap(by_true_error_type_pct: pd.DataFrame, class_names: Optional[Sequence[str]]=None, figsize=(8,5)):
    data = by_true_error_type_pct.reindex(columns=ERROR_TYPES, fill_value=0).values
    if class_names is None:
        class_names = list(by_true_error_type_pct.index)

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(data, aspect="auto")
    ax.set_xticks(range(len(ERROR_TYPES)))
    ax.set_xticklabels(ERROR_TYPES, rotation=30, ha="right")
    ax.set_yticks(range(len(class_names)))
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Error type")
    ax.set_ylabel("True class")
    ax.set_title("Error Taxonomy (% by true class)")

    # annotate
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, f"{data[i, j]:.1f}", va="center", ha="center")
    fig.tight_layout()
    return fig, ax


In [ ]:
import numpy as np, os, json
from error_taxonomy import build_error_taxonomy, plot_error_taxonomy_heatmap

# Map your variables
y_true    = np.argmax(y_test_cat, axis=1)
y_cnn     = y_pred_hybrid
cnn_proba = y_pred_proba

# Agent (LightGBM)
# If you already have:
y_agent     = y_pred
agent_proba = y_proba
# If not, compute:
# agent_proba = agent.predict(X_test_scaled)
# if agent_proba.ndim == 1:
#     agent_proba = np.vstack([1 - agent_proba, agent_proba]).T
# y_agent = np.argmax(agent_proba, axis=1)

results = build_error_taxonomy(
    y_true=y_true,
    y_cnn=y_cnn,
    y_agent=y_agent,
    cnn_proba=cnn_proba,
    agent_proba=agent_proba,
    class_names=["MES0","MES1","MES2","MES3"],
    low_conf_cutoff=0.55
)

# Save outputs
out_dir = "error_taxonomy_outputs"
os.makedirs(out_dir, exist_ok=True)
fig, ax = plot_error_taxonomy_heatmap(results["by_true_error_type_pct"])
fig.savefig(os.path.join(out_dir, "error_taxonomy_heatmap.png"), dpi=300, bbox_inches="tight")
results["by_true_error_type_pct"].round(1).to_csv(os.path.join(out_dir, "error_taxonomy_pct.csv"))
results["by_true_error_type_cnt"].to_csv(os.path.join(out_dir, "error_taxonomy_counts.csv"))
results["overrides_by_pair_cnt"].to_csv(os.path.join(out_dir, "overrides_by_pair_counts.csv"))
with open(os.path.join(out_dir, "mcnemar_table.json"), "w") as f:
    json.dump(results["mcnemar_table"], f, indent=2)

print("Saved:", out_dir)


In [ ]:
#Cohen'sKappa
from sklearn.metrics import cohen_kappa_score

# Already predicted
y_pred = clf.predict(X_test_scaled)

# Compute Kappa
kappa = cohen_kappa_score(y_test, y_pred)
print(f"Cohen’s Kappa Score: {kappa:.4f}")


In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
import matplotlib.pyplot as plt
import numpy as np

# --- Reshape image arrays ---
X_img_train_flat = X_img_train.reshape(X_img_train.shape[0], -1)
X_img_test_flat = X_img_test.reshape(X_img_test.shape[0], -1)

# --- Compute MMD with RBF kernel ---
def compute_mmd(X, Y, gamma=1e-3):
    XX = rbf_kernel(X, X, gamma=gamma)
    YY = rbf_kernel(Y, Y, gamma=gamma)
    XY = rbf_kernel(X, Y, gamma=gamma)
    return XX.mean() + YY.mean() - 2 * XY.mean()

mmd_value = compute_mmd(X_img_train_flat, X_img_test_flat, gamma=1e-3)
print(f"MMD on raw pixels: {mmd_value:.5f}")

# --- Histogram of pixel distributions ---
plt.hist(X_img_train.ravel(), bins=50, alpha=0.5, label='Train')
plt.hist(X_img_test.ravel(), bins=50, alpha=0.5, label='Test')
plt.title("Pixel Distribution Comparison")
plt.xlabel("Pixel Intensity (normalized)")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

# --- Flatten images if not already ---
X_img_train_flat = X_img_train.reshape(X_img_train.shape[0], -1)
X_img_test_flat = X_img_test.reshape(X_img_test.shape[0], -1)

# --- PCA to 2D ---
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_img_train_flat)
X_test_pca = pca.transform(X_img_test_flat)

# --- Plot ---
plt.figure(figsize=(8, 6))
markers = ['^', 's', 'o', '*']  # MES 0–3

for cls in np.unique(y_train_label):
    # Source (Train)
    idx_train = y_train_label == cls
    plt.scatter(X_train_pca[idx_train, 0], X_train_pca[idx_train, 1],
                c='green', marker=markers[cls], label=f'Source (Class {cls})', alpha=0.5)
    
    # Target (Test)
    idx_test = y_test_label == cls
    plt.scatter(X_test_pca[idx_test, 0], X_test_pca[idx_test, 1],
                c='red', marker=markers[cls], label=f'Target (Class {cls})', alpha=0.5)

plt.title(f"MMD (Raw Pixels) Visualization\nMMD = {mmd_auto:.3f} (Median Heuristic)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(loc='best')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
mmd_feat = compute_mmd(X_feat_train_scaled, X_feat_test_scaled, gamma=1.0)
print("MMD on handcrafted features:", mmd_feat)

In [ ]:
from sklearn.metrics import pairwise_distances

def median_gamma(X, Y):
    pairwise_dists = pairwise_distances(np.vstack([X, Y]))
    median_dist = np.median(pairwise_dists)
    return 1.0 / (2 * median_dist ** 2)

gamma_auto = median_gamma(X_img_train_flat, X_img_test_flat)
mmd_auto = compute_mmd(X_img_train_flat, X_img_test_flat, gamma=gamma_auto)
print("MMD (gamma from median heuristic):", mmd_auto)


In [ ]:
#ImagePertubationTest

def add_noise_to_confidence(X, std):
    X_noisy = X.copy()
    X_noisy[:, 0] += np.random.normal(0, std, size=X.shape[0])
    return X_noisy

stds = [0, 0.01, 0.03, 0.05, 0.1]
accs = []

for s in stds:
    X_noisy = add_noise_to_confidence(X_test_scaled, s)
    y_pred = clf.predict(X_noisy)
    acc = accuracy_score(y_test, y_pred)
    accs.append(acc)
    print(f"Noise STD={s:.2f} → Accuracy: {acc:.4f}")

# Plot
plt.figure()
plt.plot(stds, accs, marker='o')
plt.title("Robustness under Confidence Perturbation")
plt.xlabel("Noise Std (Confidence)")
plt.ylabel("Accuracy")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
#SHAP

import shap

explainer = shap.TreeExplainer(clf)
shap_vals = explainer.shap_values(X_test_scaled)

feature_names = ["confidence", "umap_0", "umap_1"] + [f"f{i}" for i in range(20)]

# Summary plot
shap.summary_plot(shap_vals, X_test_scaled, feature_names=feature_names)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === Step 1: Transpose SHAP array if needed ===
if shap_vals.shape != (4, X_test_scaled.shape[0], len(feature_names)):
    shap_vals_arr = shap_vals.transpose((2, 0, 1))  # (4, samples, features)
else:
    shap_vals_arr = shap_vals

# === Step 2: Compute mean absolute SHAP value per class per feature ===
mean_abs = np.abs(shap_vals_arr).mean(axis=1)  # shape: (n_classes, n_features)

# === Step 3: Normalize each row to % contribution ===
mean_abs_pct = (mean_abs.T / mean_abs.sum(axis=1)).T * 100

# === Step 4: Plot grouped horizontal bar chart ===
classes = [f"MES {i}" for i in range(4)]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']  # Blue, Orange, Green, Red

fig, ax = plt.subplots(figsize=(12, 10))

bar_width = 0.2
y_pos = np.arange(len(feature_names))

for i in range(4):
    offset = (i - 1.5) * bar_width  # Shift bars side by side
    ax.barh(
        y_pos + offset,
        mean_abs_pct[i],
        height=bar_width,
        label=classes[i],
        color=colors[i]
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(feature_names)
ax.invert_yaxis()
ax.set_xlabel("Mean Absolute SHAP Contribution (%)")
ax.set_title("Feature Importance by Class (SHAP Values)")
ax.legend(title="Predicted Class")
plt.tight_layout()

# Save the figure
plt.savefig(os.path.join(save_dir, "shap_grouped_bar_by_class.png"))
plt.show()


In [ ]:
import numpy as np
import pandas as pd

# === Step 1: Ensure SHAP shape is (classes, samples, features) ===
if shap_vals.shape != (4, X_test_scaled.shape[0], len(feature_names)):
    shap_vals_arr = shap_vals.transpose((2, 0, 1))  # (4, samples, features)
else:
    shap_vals_arr = shap_vals

# === Step 2: Compute mean absolute SHAP values globally ===
global_mean_shap = np.abs(shap_vals_arr).mean(axis=(0, 1))  # shape: (features,)

# === Step 3: Convert to percentage ===
shap_total = global_mean_shap.sum()
shap_pct = (global_mean_shap / shap_total) * 100

# === Step 4: Create and sort table ===
shap_global_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean SHAP Value": global_mean_shap,
    "SHAP % Contribution": shap_pct
}).sort_values(by="SHAP % Contribution", ascending=False).reset_index(drop=True)

# === Display in notebook ===
display(shap_global_df.round(3))


In [ ]:
import matplotlib.pyplot as plt

# Sort the dataframe from highest to lowest SHAP % contribution
shap_global_sorted = shap_global_df.sort_values(by="SHAP % Contribution", ascending=False)

# Plot
plt.figure(figsize=(10, 6))
shap_global_sorted.plot(
    kind='barh',
    x='Feature',
    y='SHAP % Contribution',
    legend=False,
    color='skyblue'
)

plt.title("Global SHAP Feature Importance")
plt.xlabel("SHAP % Contribution")
plt.ylabel("Feature")
plt.gca().invert_yaxis()  # Show highest on top
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd

# Your SHAP values shape: (samples, features, classes)
shap_vals_arr = np.array(shap_vals)  # shape: (9127, 23, 4)

# Transpose to: (classes, samples, features)
shap_vals_arr = shap_vals_arr.transpose((2, 0, 1))  # shape → (4, 9127, 23)

# Now we can process normally
n_classes, n_samples, n_features = shap_vals_arr.shape

# === Feature names (23) ===
feature_names = [
    "confidence", "umap_0", "umap_1",
    "LL_Mean", "LL_Std", "LL_Var", "LL_Entropy",
    "LH_Mean", "LH_Std", "LH_Var", "LH_Entropy",
    "HL_Mean", "HL_Std", "HL_Var", "HL_Entropy",
    "HH_Mean", "HH_Std", "HH_Var", "HH_Entropy",
    "HH_Energy",
    "GLCM_Contrast", "GLCM_Dissimilarity", "GLCM_Homogeneity"
]

# === Compute absolute mean and % per class ===
mean_abs = np.abs(shap_vals_arr).mean(axis=1)  # shape: (n_classes, n_features)
mean_abs_pct = (mean_abs.T / mean_abs.sum(axis=1)).T * 100  # normalized %

# === Build table ===
shap_pct_df = pd.DataFrame(mean_abs_pct, columns=feature_names)
shap_pct_df.index = [f"MES {i}" for i in range(n_classes)]

# === Display full SHAP % table ===
print("📊 SHAP Feature Contribution (%) per Class")
display(shap_pct_df.round(2))

# Print top 5 SHAP features for each class
print("\n Top 5 SHAP Features per Class (% contribution):\n")

for i in range(n_classes):
    class_name = f"MES {i}"
    top_idx = np.argsort(-mean_abs[i])[:5]
    for rank, idx in enumerate(top_idx):
        fname = feature_names[idx]
        pct = mean_abs_pct[i, idx]
        print(f"{class_name:<6} #{rank+1}: {fname:<25} → {pct:>5.2f}%")
    print()



In [ ]:
print(f"SHAP shape: {np.array(shap_vals).shape}")


In [ ]:
#FGSMAttack
epsilon = 0.05  # max allowed change
X_adv = X_test_scaled.copy()
X_adv[:, 0] += epsilon  # perturb confidence

# Predict and evaluate
y_pred_adv = clf.predict(X_adv)
adv_acc = accuracy_score(y_test, y_pred_adv)
print(f"Adversarial Accuracy (ε={epsilon} on confidence): {adv_acc:.4f}")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2

# --- Select one image ---
idx = 2100
orig_img = X_img_test[idx]
true_label = y_test_label[idx]

x_img = tf.convert_to_tensor(orig_img[np.newaxis], dtype=tf.float32)  # shape (1, 224, 224, 3)
x_handcrafted = tf.convert_to_tensor(np.zeros((1, 20), dtype=np.float32))  # dummy handcrafted input
x_umap = tf.convert_to_tensor(np.zeros((1, 2), dtype=np.float32))  # dummy umap input

# === FGSM Perturbation ===
with tf.GradientTape() as tape:
    tape.watch(x_img)
    pred = model_hybrid([x_img, x_handcrafted, x_umap])
    loss = tf.keras.losses.categorical_crossentropy(y_test_cat[idx:idx+1], pred)

epsilon = 0.05
grad = tape.gradient(loss, x_img)
signed_grad = tf.sign(grad)
perturbed_img = tf.clip_by_value(x_img + epsilon * signed_grad, 0, 1).numpy()[0]

# === Extract handcrafted features ===
def compute_all_features(img_uint8):
    wavelet_feats = extract_wavelet_stats(img_uint8)
    glcm_feats = extract_glcm_features_extended(img_uint8)
    return wavelet_feats + glcm_feats

# Extract handcrafted features (uint8 required)
orig_feat = compute_all_features((orig_img * 255).astype(np.uint8))
pert_feat = compute_all_features((perturbed_img * 255).astype(np.uint8))

# Pad to 23D input: [confidence, umap_0, umap_1, handcrafted]
orig_input = np.hstack(([0, 0, 0], orig_feat))
pert_input = np.hstack(([0, 0, 0], pert_feat))

# Scale with pre-fitted scaler
orig_scaled = scaler.transform([orig_input])
pert_scaled = scaler.transform([pert_input])

# Agent prediction using Booster
orig_pred = agent.predict(orig_scaled)  # shape: (n_classes,)
pert_pred = agent.predict(pert_scaled)

# --- Plot: Original | Perturbed | Perturbation Heatmap ---
plt.figure(figsize=(10, 3))

plt.subplot(1, 3, 1)
plt.imshow(orig_img)
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(perturbed_img)
plt.title("FGSM Perturbed")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(np.abs(orig_img - perturbed_img))
plt.title("Perturbation Heatmap")
plt.axis('off')

plt.tight_layout()
plt.show()

# --- Result Output ---
print(f"✅ True Label        : {true_label}")
print(f"🧠 Agent (Original)  : MES {np.argmax(orig_pred)}  | {np.round(orig_pred, 3)}")
print(f"⚠️ Agent (Perturbed) : MES {np.argmax(pert_pred)}  | {np.round(pert_pred, 3)}")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from PIL import Image

# -----------------------------
# ✅ CONFIG: Set your dataset root folder
# -----------------------------
base_path = "/home/D13K48009/Clara/Dataset"  # ← CHANGE THIS
center_folders = {
    "Center 1": os.path.join(base_path, "MES classification_20250313"),
    "Center 2": os.path.join(base_path, "MES classification_20250724"),
    "Center 3": os.path.join(base_path, "MES Mixed Data"),
    "Center 4": os.path.join(base_path, "MES_Colonoscopy Public Dataset"),
}

# -----------------------------
# ✅ Helper: Load all pixels from all images in one center
# -----------------------------
def load_all_pixels_from_center(folder):
    all_pixels = []
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff', '*.tif']
    for class_id in range(4):
        class_folder = os.path.join(folder, f"MES{class_id}")
        if not os.path.exists(class_folder):
            continue
        img_paths = []
        for ext in extensions:
            img_paths.extend(glob(os.path.join(class_folder, ext)))
        for img_path in img_paths:
            try:
                img = Image.open(img_path).convert('L')  # Grayscale
                img = np.array(img) / 255.0
                all_pixels.extend(img.flatten())
            except Exception as e:
                print(f"[ERROR] Skipped {img_path}: {e}")
    return np.array(all_pixels)

# -----------------------------
# ✅ Helper: Select one image per MES class (0–3)
# -----------------------------
def select_sample_per_class(folder):
    selected_images = {}
    extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff', '*.tif']
    for class_id in range(4):
        class_folder = os.path.join(folder, f"MES{class_id}")
        if not os.path.exists(class_folder):
            continue
        img_paths = []
        for ext in extensions:
            img_paths.extend(glob(os.path.join(class_folder, ext)))
        if img_paths:
            selected_images[class_id] = img_paths[0]  # Pick first found image
    return selected_images

# -----------------------------
# ✅ Plot 1: Pixel intensity distribution per center
# -----------------------------
plt.figure(figsize=(8, 6))
for center_name, folder in center_folders.items():
    pixels = load_all_pixels_from_center(folder)
    if len(pixels) == 0:
        print(f"[WARNING] No pixels loaded for {center_name}")
        continue
    sns.kdeplot(pixels, label=center_name, fill=True, alpha=0.4)
plt.title("Per-center Normalized Pixel Intensity Distribution")
plt.xlabel("Pixels")
plt.xticks(np.arange(0, 1.01, 0.1))  # Add tick marks every 0.1
plt.ylabel("Distribution")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Plot 2: One image per class with histogram
# -----------------------------
for center_name, folder in center_folders.items():
    print(f"\n Showing samples for {center_name}")
    selected_images = select_sample_per_class(folder)

    fig, axs = plt.subplots(2, 4, figsize=(16, 4))
    for class_id in range(4):
        if class_id in selected_images:
            img = Image.open(selected_images[class_id]).convert('L')
            img_np = np.array(img)
            img_norm = img_np / 255.0

            axs[0, class_id].imshow(img_np, cmap='gray')
            axs[0, class_id].axis('off')
            axs[0, class_id].set_title(f'MES {class_id}')

            axs[1, class_id].hist(img_norm.flatten(), bins=50, color='steelblue')
            axs[1, class_id].set_xlim(0, 1)
            axs[1, class_id].set_xlabel("Intensity")
            axs[1, class_id].set_ylabel("Count")
        else:
            axs[0, class_id].set_visible(False)
            axs[1, class_id].set_visible(False)

    fig.suptitle(f"{center_name} - Image and Histogram per Class")
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from PIL import Image

# -----------------------------
# ✅ CONFIG
# -----------------------------
base_path = "/home/D13K48009/Clara/Dataset"
center_folders = {
    "Center 1": os.path.join(base_path, "MES classification_20250313"),
    "Center 2": os.path.join(base_path, "MES classification_20250724"),
    "Center 3": os.path.join(base_path, "MES Mixed Data"),
    "Center 4": os.path.join(base_path, "MES_Colonoscopy Public Dataset"),
}
extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff', '*.tif']

# -----------------------------
# ✅ Load grayscale pixels from all images
# -----------------------------
def load_all_pixels_from_center(folder):
    all_pixels = []
    for class_id in range(4):
        class_folder = os.path.join(folder, f"MES{class_id}")
        if not os.path.exists(class_folder):
            continue
        img_paths = []
        for ext in extensions:
            img_paths.extend(glob(os.path.join(class_folder, ext)))
        for img_path in img_paths:
            try:
                img = Image.open(img_path).convert('L')
                img = np.array(img) / 255.0
                all_pixels.extend(img.flatten())
            except Exception as e:
                print(f"[ERROR] Skipped {img_path}: {e}")
    return np.array(all_pixels)

# -----------------------------
# ✅ Select one RGB image per MES class
# -----------------------------
def select_sample_per_class(folder):
    selected_images = {}
    for class_id in range(4):
        class_folder = os.path.join(folder, f"MES{class_id}")
        if not os.path.exists(class_folder):
            continue
        img_paths = []
        for ext in extensions:
            img_paths.extend(glob(os.path.join(class_folder, ext)))
        if img_paths:
            selected_images[class_id] = img_paths[0]  # Take first image
    return selected_images

# -----------------------------
# ✅ Plot 1: Improved grayscale KDE distribution (log-scale, zoomed)
# -----------------------------
plt.figure(figsize=(10, 6))
for center_name, folder in center_folders.items():
    pixels = load_all_pixels_from_center(folder)
    if len(pixels) == 0:
        print(f"[WARNING] No pixels loaded for {center_name}")
        continue
    sns.kdeplot(pixels, label=center_name, lw=2)
plt.title("Per center normalization (Grayscale Intensities)")
plt.xlabel("Normalized Pixel Intensities")
plt.ylabel("Density (log scale)")
plt.yscale("log")
plt.xlim(0, 0.3)  # Focus on main region of intensity
plt.legend()
plt.tight_layout()
plt.show()

# -----------------------------
# ✅ Plot 2: RGB image + RGB histogram per class
# -----------------------------
for center_name, folder in center_folders.items():
    print(f"\n Showing RGB samples for {center_name}")
    selected_images = select_sample_per_class(folder)

    fig, axs = plt.subplots(2, 4, figsize=(20, 6))
    for class_id in range(4):
        if class_id in selected_images:
            img = Image.open(selected_images[class_id]).convert('RGB')
            img_np = np.array(img)
            axs[0, class_id].imshow(img_np)
            axs[0, class_id].axis('off')
            axs[0, class_id].set_title(f'MES {class_id}')

            # Plot R, G, B histograms
            for i, (color, label) in enumerate(zip(['r', 'g', 'b'], ['Red', 'Green', 'Blue'])):
                channel_data = img_np[..., i].flatten()
                axs[1, class_id].hist(channel_data, bins=256, color=color, alpha=0.4, label=label, density=True)
            axs[1, class_id].set_xlim(0, 255)
            axs[1, class_id].set_xlabel("Pixel Value")
            axs[1, class_id].set_ylabel("Density")
            axs[1, class_id].legend()
        else:
            axs[0, class_id].set_visible(False)
            axs[1, class_id].set_visible(False)

    fig.suptitle(f"{center_name} - RGB Image and RGB Channel Histogram per Class", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
# -----------------------------
# ✅ Plot 2: RGB image + RGB histogram per class
# -----------------------------
for center_name, folder in center_folders.items():
    print(f"\n Showing RGB samples for {center_name}")
    selected_images = select_sample_per_class(folder)

    fig, axs = plt.subplots(2, 4, figsize=(20, 6))
    for class_id in range(4):
        if class_id in selected_images:
            img = Image.open(selected_images[class_id]).convert('RGB')
            img_np = np.array(img)
            axs[0, class_id].imshow(img_np)
            axs[0, class_id].axis('off')
            axs[0, class_id].set_title(f'MES {class_id}')

            # Plot R, G, B histograms
            for i, (color, label) in enumerate(zip(['r', 'g', 'b'], ['Red', 'Green', 'Blue'])):
                channel_data = img_np[..., i].flatten()
                axs[1, class_id].hist(channel_data, bins=256, color=color, alpha=0.4, label=label, density=True)
            axs[1, class_id].set_xlim(0, 255)
            axs[1, class_id].set_xlabel("Pixel Value")
            axs[1, class_id].set_ylabel("Density")
            axs[1, class_id].legend()
        else:
            axs[0, class_id].set_visible(False)
            axs[1, class_id].set_visible(False)

    fig.suptitle(f"{center_name} - RGB Image and RGB Channel Histogram per Class", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from PIL import Image
from sklearn.metrics.pairwise import pairwise_distances, rbf_kernel

# -----------------------------
# CONFIG: Set your folder paths
# -----------------------------
base_path = "/home/D13K48009/Clara/Dataset"
center_folders = {
    "Center 1": os.path.join(base_path, "MES classification_20250313"),
    "Center 2": os.path.join(base_path, "MES classification_20250724"),
    "Center 3": os.path.join(base_path, "MES Mixed Data"),
    "Center 4": os.path.join(base_path, "MES_Colonoscopy Public Dataset"),
}
extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff', '*.tif']

# -----------------------------
# STEP 1: Load all grayscale pixels from one center
# -----------------------------
def load_pixels(folder, max_images=100):  # limit for speed
    all_pixels = []
    for class_id in range(4):
        class_folder = os.path.join(folder, f"MES{class_id}")
        if not os.path.exists(class_folder):
            continue
        img_paths = []
        for ext in extensions:
            img_paths.extend(glob(os.path.join(class_folder, ext)))
        img_paths = img_paths[:max_images]
        for img_path in img_paths:
            try:
                img = Image.open(img_path).convert('L')
                img = img.resize((64, 64))  # speed-up
                img_np = np.array(img) / 255.0
                all_pixels.append(img_np.flatten())
            except:
                continue
    return np.array(all_pixels)

# -----------------------------
# STEP 2: Compute MMD using median heuristic
# -----------------------------
def compute_mmd(X, Y, gamma=None):
    if gamma is None:
        all_samples = np.vstack([X, Y])
        dists = pairwise_distances(all_samples)
        median = np.median(dists)
        gamma = 1.0 / (2 * median**2)
    K_XX = rbf_kernel(X, X, gamma=gamma)
    K_YY = rbf_kernel(Y, Y, gamma=gamma)
    K_XY = rbf_kernel(X, Y, gamma=gamma)
    m = X.shape[0]
    n = Y.shape[0]
    mmd_sq = (np.sum(K_XX) / (m * m) +
              np.sum(K_YY) / (n * n) -
              2 * np.sum(K_XY) / (m * n))
    return np.sqrt(mmd_sq)

# -----------------------------
# STEP 3: Compute pairwise MMD matrix
# -----------------------------
center_names = list(center_folders.keys())
pixel_data = {name: load_pixels(path) for name, path in center_folders.items()}
mmd_matrix = np.zeros((len(center_names), len(center_names)))

for i in range(len(center_names)):
    for j in range(len(center_names)):
        X = pixel_data[center_names[i]]
        Y = pixel_data[center_names[j]]
        if len(X) == 0 or len(Y) == 0:
            mmd_matrix[i, j] = np.nan
            continue
        mmd_score = compute_mmd(X, Y)
        mmd_matrix[i, j] = mmd_score

# -----------------------------
# STEP 4: Plot heatmap
# -----------------------------
plt.figure(figsize=(8, 6))
sns.heatmap(mmd_matrix, xticklabels=center_names, yticklabels=center_names,
            annot=True, fmt=".3f", cmap="coolwarm", linewidths=0.5)
plt.title("MMD Distance Matrix Between Centers (Grayscale Feature Space)")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Define MMD values (upper triangle only)
center_names = ["Center 1", "Center 2", "Center 3", "Center 4"]
mmd_values = np.array([
    [0.000, 0.093, 0.053, 0.620],
    [0.093, 0.000, 0.050, 0.598],
    [0.053, 0.050, 0.000, 0.609],
    [0.620, 0.598, 0.609, 0.000]
])

# Convert to long format DataFrame
data = []
for i in range(4):
    for j in range(i+1, 4):
        data.append({
            "From": center_names[i],
            "To": center_names[j],
            "MMD": mmd_values[i, j]
        })
df_long = pd.DataFrame(data)

# Plot
plt.figure(figsize=(8, 4))
sns.barplot(data=df_long, x="From", y="MMD", hue="To", palette="Blues")
plt.title("Pairwise MMD Distances Between Centers")
plt.ylabel("MMD Score")
plt.xlabel("Reference Center")
plt.ylim(0, max(df_long["MMD"]) + 0.1)
plt.legend(title="Compared To")
plt.tight_layout()
plt.savefig("mmd_barplot.png", dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
for i in range(4):
    plt.plot(center_names, mmd_values[i], marker='o', label=center_names[i])

plt.title("MMD Profile of Each Center")
plt.xlabel("Compared To")
plt.ylabel("MMD Score")
plt.legend(title="Reference Center")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("mmd_lineplot.png", dpi=300)
plt.show()